In [ ]:
import arviz as az
import arviz_plots as azp
from arviz_base import load_arviz_data
azp.style.use("arviz-variat")
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive

## Generate fake data

In [ ]:
key = jax.random.PRNGKey(42)
n = 50

true_intercept = 2.0
true_slope = 3.0
true_sigma = 3.0

key, sub_key = jax.random.split(key)
x = jax.random.normal(sub_key, shape=(n,))
key, sub_key = jax.random.split(key)
y = (
    true_slope * x
    + true_intercept
    + true_sigma * jax.random.normal(sub_key, shape=(n,))
)

plt.figure()
plt.scatter(x, y, label="Data", color="k")
plt.legend()
plt.show()

## Define model

In [ ]:
def model(x: jax.Array, y: jax.Array | None = None) -> None:
    intercept = numpyro.sample("intercept", dist.Normal(0, 10))

    slope = numpyro.sample("slope", dist.Normal(0, 10))

    sigma = numpyro.sample("sigma", dist.HalfNormal(5))

    mu = intercept + slope * x

    with numpyro.plate("data", len(x)):
        numpyro.sample("obs", dist.Normal(mu, sigma), obs=y)

In [ ]:
numpyro.render_model(model, model_kwargs={"x": x}, render_distributions=True)

## Doing inference

In [ ]:
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=10000)
key, sub_key = jax.random.split(key)
mcmc.run(sub_key, x=x, y=y)
mcmc.print_summary()

In [ ]:
posterior_samples = mcmc.get_samples()
idata = az.from_numpyro(mcmc)

In [ ]:
plt.figure()
az.plot_dist(idata, var_names=["intercept", "slope", "sigma"])
azp.plot_pair(idata, var_names=["intercept", "slope", "sigma"])
plt.gcf().set_size_inches(8, 8)
plt.show()

In [ ]:
x_plot = jnp.linspace(x.min(), x.max(), 100)
y_true = true_intercept + true_slope * x_plot
n_lines = 1000

key, subkey = jax.random.split(key)
idx = jax.random.choice(
    subkey, len(posterior_samples["slope"]), shape=(n_lines,), replace=False
)

plt.figure()
for i in idx:
    intercept_i = posterior_samples["intercept"][i]
    slope_i = posterior_samples["slope"][i]
    y_line = intercept_i + slope_i * x_plot
    plt.plot(x_plot, y_line, alpha=0.01, color="red")
plt.plot(x_plot, y_true, linewidth=2, label="True regression line", color="black")
plt.legend()
plt.show()

## Sample new points

In [ ]:
key, sub_key = jax.random.split(key)
posterior_predictive = Predictive(model, posterior_samples=posterior_samples)
predictive_samples = posterior_predictive(sub_key, x=x)
y_new = predictive_samples["obs"][0, :]

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(x, y, color="black", label="Observed data")
plt.scatter(x, y_new, color="red", alpha=0.5, label="Posterior predictive sample")

plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## Diagnostics - Trace plots

In [ ]:

plt.figure()
azp.plot_trace(idata, )
plt.show()

## Running analysis with more chains

In [ ]:
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=10000, num_chains=4)
key, sub_key = jax.random.split(key)
mcmc.run(sub_key, x=x, y=y)
mcmc.print_summary()

In [ ]:
idata = az.from_numpyro(mcmc)

In [ ]:
plt.figure()
azp.plot_trace(idata)
plt.show()

## Diagnostics - Effective sample size

In [ ]:
plt.figure()
azp.plot_ess_evolution(idata)
plt.show()
# Shows mean and deviation of ESS across the four chains

In [ ]:
plt.figure()
azp.plot_ess(idata, extra_methods=True)
plt.show()
# Shows mean and deviation of ESS across all quantiles. Stable and efficient ESS across all quantiles indicate good mixing and convergence of the chains.

## Diagnostics - Monte Carlo Standard Error

In [ ]:
plt.figure()
azp.plot_mcse(idata, extra_methods=True)
for ax in plt.gcf().axes:
    var_name = ax.get_title()
    samples = idata.posterior[var_name]

    posterior_mean = samples.mean().item()
    posterior_sd = samples.std().item()

    ax.set_title(
        f"{var_name}\nposterior mean = {posterior_mean:.2f}, SD = {posterior_sd:.2f}"
    )

plt.show()
# The horizontal dashed lines are reference MCSEs for the parameter’s posterior mean and standard deviation.